# Merging data files and converting to indices for fast access


In [3]:
import h5py
import os
import numpy as np
from tqdm import tqdm  # specific for progress bars, optional

# --- Configuration ---
# Directory where your foldX files are located
base_dir = '/project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/' 

# The destination file (from your previous snippet)
output_path = '/project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/merged_tile_data.hdf5'

# Number of folds
num_folds = 5

# Create directory for output if it doesn't exist
output_dir = os.path.dirname(output_hdf5_path)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- merging Logic ---

# --- Main Script ---
with h5py.File(output_path, 'w') as f_out:
    print(f"Created output file: {output_path}")
    
    total_patients = 0
    
    # Loop through the 5 folds
    for i in range(5):
        input_filename = f'tile_preds_fold{i}.hdf5'
        input_full_path = os.path.join(base_dir, input_filename)
        
        if not os.path.exists(input_full_path):
            print(f"Warning: {input_filename} missing. Skipping.")
            continue

        print(f"Processing {input_filename}...")
        
        with h5py.File(input_full_path, 'r') as f_in:
            keys = list(f_in.keys())
            total_patients += len(keys)
            
            for patient_id in tqdm(keys, desc=f"Fold {i}"):
                # .copy() efficiently transfers the data structure without 
                # loading numpy arrays into memory
                f_in.copy(patient_id, f_out)

print(f"\nSuccess! Merged {total_patients} unique patients into {output_path}")

Created output file: /project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/merged_tile_data.hdf5
Processing tile_preds_fold0.hdf5...


Fold 0: 100%|██████████| 78/78 [01:46<00:00,  1.37s/it]


Processing tile_preds_fold1.hdf5...


Fold 1: 100%|██████████| 77/77 [01:53<00:00,  1.48s/it]


Processing tile_preds_fold2.hdf5...


...

Processing tile_preds_fold3.hdf5...


Fold 3: 100%|██████████| 77/77 [01:37<00:00,  1.26s/it]


Processing tile_preds_fold4.hdf5...


Fold 4: 100%|██████████| 77/77 [02:07<00:00,  1.66s/it]



Success! Merged 386 unique patients into /project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/merged_tile_data.hdf5


In [4]:
import h5py
import numpy as np
import re


input_hdf5_path = '/project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/merged_tile_data.hdf5'
output_npy_path = '/project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/GS_merged_tile_indices.npy'

gleason_pattern = re.compile(r'GS(\d)\+(\d)')

indices = []

# Open the HDF5 file
with h5py.File(input_hdf5_path, 'r') as f:
    for patient_id in f.keys():
        patient_group = f[patient_id]
        for tile_id in patient_group.keys():
            match = gleason_pattern.search(tile_id)
            if match:
                gs_label= match.group()
                indices.append((patient_id, tile_id, gs_label))

# Convert to numpy array for easy saving/loading
# Use dtype=object because patient_id and tile_id are strings
indices_array = np.array(indices, dtype=object)

np.save(output_npy_path, indices_array)

print(f"Saved {len(indices)} indices to {output_npy_path}")

Saved 718958 indices to /project_antwerp/TCGA-PRAD/outputs/TCGA/PRAD/GS_merged_tile_indices.npy
